# 가설 검증: 선호 카테고리 쿠폰은 이탈 위험 고객의 재구매를 유도하는가?

## 가설
> 평소 구매주기보다 1.5배 이상 구매가 지연된 고객에게 과거 주 구매 카테고리와 일치하는 쿠폰 캠페인을 제공하면, 캠페인을 제공하지 않은 유사 고객보다 28일 이내 재구매율이 높을 것이다.

- H0: 두 집단의 28일 재구매율 차이는 0 이하이다.
- H1: 선호 카테고리 일치 쿠폰 집단의 28일 재구매율이 더 높다.

## 고정 기준
- 이탈 위험: 캠페인 시작 당시 최근 구매 경과일 ÷ 개인 중앙 구매간격 > 1.5
- 구매주기: 시작 전 최대 26주, 서로 다른 구매일 최소 3개
- 선호 카테고리: 시작 전 12주 매출 상위 3개 카테고리
- 일치 쿠폰: 캠페인 쿠폰 상품 카테고리와 선호 카테고리가 하나 이상 일치
- 결과: 캠페인 시작 후 28일 이내 재구매 여부와 28일 매출
- 비교군: 같은 시점의 유사 위험 고객 중 전후 28일 동안 캠페인 미수신
- 캠페인 품질: 쿠폰 카테고리가 50개 이하인 캠페인만 사용. 지나치게 광범위한 캠페인은 ‘개인 선호 일치’가 거의 자동으로 성립하므로 제외

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns",60)
PROJECT_DIR=Path.cwd().parent if Path.cwd().name=="notebooks" else Path.cwd()
OUTPUT_DIR=PROJECT_DIR/"data"/"processed"
FOLLOWUP_DAYS=28
PREFERENCE_DAYS=84
CYCLE_DAYS=182
MAX_CAMPAIGN_CATEGORIES=50
MIN_TREATED=10
RANDOM_SEED=42
DATASET_END_DAY=711

## 1. 거래·상품·캠페인·쿠폰 데이터 연결

In [ ]:
tx=pd.read_csv(PROJECT_DIR/"transaction_data.csv",usecols=["household_key","BASKET_ID","DAY","PRODUCT_ID","SALES_VALUE"],
    dtype={"household_key":"int32","BASKET_ID":"int64","DAY":"int16","PRODUCT_ID":"int32","SALES_VALUE":"float32"})
products=pd.read_csv(PROJECT_DIR/"product.csv",usecols=["PRODUCT_ID","COMMODITY_DESC"])
campaign_desc=pd.read_csv(PROJECT_DIR/"campaign_desc.csv")
campaign_table=pd.read_csv(PROJECT_DIR/"campaign_table.csv")
coupons=pd.read_csv(PROJECT_DIR/"coupon.csv").drop_duplicates()
coupon_categories=coupons.merge(products,on="PRODUCT_ID",how="left").dropna(subset=["COMMODITY_DESC"])
campaign_category_count=coupon_categories.groupby("CAMPAIGN")["COMMODITY_DESC"].nunique().rename("coupon_categories")
valid_campaigns=campaign_desc.loc[campaign_desc["START_DAY"]+FOLLOWUP_DAYS-1<=DATASET_END_DAY].copy()
valid_campaigns=valid_campaigns.merge(campaign_category_count,on="CAMPAIGN",how="left")
valid_campaigns=valid_campaigns.loc[valid_campaigns["coupon_categories"].between(1,MAX_CAMPAIGN_CATEGORIES)].copy()
all_households=np.sort(tx["household_key"].unique())
print(f"분석 가능 캠페인: {len(valid_campaigns)}개")
display(valid_campaigns[["CAMPAIGN","DESCRIPTION","START_DAY","coupon_categories"]].sort_values("START_DAY"))

## 2. 캠페인 시작일 기준 위험도·선호 카테고리·성과 생성

In [ ]:
def customer_features(start_day):
    pref_start=max(1,start_day-PREFERENCE_DAYS)
    pre=tx.loc[tx["DAY"].between(pref_start,start_day-1)]
    value=pre.groupby("household_key").agg(pre_revenue=("SALES_VALUE","sum"),pre_baskets=("BASKET_ID","nunique"))
    value=value.reindex(all_households,fill_value=0)
    active=value["pre_baskets"]>0
    value["value_score"]=0.0
    value.loc[active,"value_score"]=(value.loc[active,"pre_revenue"].rank(pct=True)+value.loc[active,"pre_baskets"].rank(pct=True))/2

    cycle_start=max(1,start_day-CYCLE_DAYS)
    visits=tx.loc[tx["DAY"].between(cycle_start,start_day-1),["household_key","DAY"]].drop_duplicates().sort_values(["household_key","DAY"])
    visits["gap"]=visits.groupby("household_key")["DAY"].diff()
    cycle=visits.groupby("household_key").agg(visit_days=("DAY","nunique"),last_day=("DAY","max"),median_gap=("gap","median"))
    features=value.join(cycle,how="left").reset_index()
    features["recency"]=start_day-features["last_day"]
    features["gap_ratio"]=features["recency"]/features["median_gap"].replace(0,np.nan)
    features.loc[features["visit_days"]<3,"gap_ratio"]=np.nan
    features["at_risk"]=features["gap_ratio"]>1.5

    pref=pre.merge(products,on="PRODUCT_ID",how="left").groupby(["household_key","COMMODITY_DESC"],dropna=False)["SALES_VALUE"].sum().reset_index(name="category_revenue")
    pref["preference_rank"]=pref.groupby("household_key")["category_revenue"].rank(method="first",ascending=False)
    top3=pref.loc[pref["preference_rank"]<=3,["household_key","COMMODITY_DESC","preference_rank"]]
    return features,top3

def outcomes(start_day):
    follow_end=start_day+FOLLOWUP_DAYS-1
    out=tx.loc[tx["DAY"].between(start_day,follow_end)].groupby("household_key").agg(
        revenue_28d=("SALES_VALUE","sum"),baskets_28d=("BASKET_ID","nunique")
    ).reindex(all_households,fill_value=0).reset_index()
    out["repurchase_28d"]=(out["baskets_28d"]>0).astype(int)
    return out

## 3. 선호 일치 쿠폰 고객과 무캠페인 고객 매칭

비교군은 구매주기 지연, 직전 매출·구매횟수, 고객가치가 가장 가까운 고객을 캠페인별로 선택합니다.

In [ ]:
def nearest_controls(treated,controls):
    cols=["gap_ratio","pre_revenue","pre_baskets","value_score"]
    both=pd.concat([treated[cols],controls[cols]])
    x=np.log1p(both[cols].clip(lower=0).to_numpy(float))
    mean,std=x.mean(axis=0),x.std(axis=0); std[std==0]=1
    t=(np.log1p(treated[cols].clip(lower=0).to_numpy(float))-mean)/std
    c=(np.log1p(controls[cols].clip(lower=0).to_numpy(float))-mean)/std
    dist=((t[:,None,:]-c[None,:,:])**2).sum(axis=2)
    return controls.iloc[dist.argmin(axis=1)].reset_index(drop=True)

def smd(a,b):
    pooled=np.sqrt((a.var(ddof=1)+b.var(ddof=1))/2)
    return 0 if pooled==0 else (a.mean()-b.mean())/pooled

## 4. 캠페인별 28일 재구매 효과 계산

In [ ]:
campaign_results,pair_results=[],[]
for camp in valid_campaigns.sort_values("CAMPAIGN").itertuples(index=False):
    cid,start=int(camp.CAMPAIGN),int(camp.START_DAY); follow_end=start+FOLLOWUP_DAYS-1
    features,top3=customer_features(start); data=features.merge(outcomes(start),on="household_key",how="left")
    camp_categories=set(coupon_categories.loc[coupon_categories["CAMPAIGN"]==cid,"COMMODITY_DESC"])
    matched_pref_ids=set(top3.loc[top3["COMMODITY_DESC"].isin(camp_categories),"household_key"])
    assigned_ids=set(campaign_table.loc[campaign_table["CAMPAIGN"]==cid,"household_key"])
    treated_ids=assigned_ids & matched_pref_ids
    overlaps=campaign_desc.loc[(campaign_desc["START_DAY"]<=follow_end)&(campaign_desc["END_DAY"]>=start-FOLLOWUP_DAYS),"CAMPAIGN"]
    exposed_ids=set(campaign_table.loc[campaign_table["CAMPAIGN"].isin(overlaps),"household_key"])
    eligible=data.loc[data["at_risk"]&data["gap_ratio"].notna()].copy()
    treated=eligible.loc[eligible["household_key"].isin(treated_ids)].reset_index(drop=True)
    controls=eligible.loc[~eligible["household_key"].isin(exposed_ids)].reset_index(drop=True)
    assigned_risk=eligible["household_key"].isin(assigned_ids).sum()
    if len(treated)<MIN_TREATED or len(controls)<MIN_TREATED: continue
    matched=nearest_controls(treated,controls)
    rep_effect=treated["repurchase_28d"].to_numpy()-matched["repurchase_28d"].to_numpy()
    rev_effect=treated["revenue_28d"].to_numpy()-matched["revenue_28d"].to_numpy()
    balance_gap=smd(treated["gap_ratio"],matched["gap_ratio"]); balance_rev=smd(treated["pre_revenue"],matched["pre_revenue"])
    campaign_results.append({
        "campaign":cid,"campaign_type":camp.DESCRIPTION,"coupon_categories":camp.coupon_categories,
        "assigned_at_risk":int(assigned_risk),"matched_coupon_at_risk":len(treated),"unique_controls":matched["household_key"].nunique(),
        "treated_repurchase_rate":treated["repurchase_28d"].mean(),"control_repurchase_rate":matched["repurchase_28d"].mean(),
        "repurchase_rate_effect":rep_effect.mean(),"revenue_28d_effect":rev_effect.mean(),
        "gap_ratio_balance_smd":balance_gap,"pre_revenue_balance_smd":balance_rev,
    })
    pair_results.append(pd.DataFrame({"campaign":cid,"treated_household":treated["household_key"],
        "control_household":matched["household_key"],"repurchase_effect":rep_effect,"revenue_28d_effect":rev_effect}))

personalized_coupon_campaign_effects=pd.DataFrame(campaign_results)
personalized_coupon_pairs=pd.concat(pair_results,ignore_index=True) if pair_results else pd.DataFrame()
personalized_coupon_campaign_effects["quality_pass"]=(personalized_coupon_campaign_effects[["gap_ratio_balance_smd","pre_revenue_balance_smd"]].abs().max(axis=1)<=.1)
display(personalized_coupon_campaign_effects)

## 5. 캠페인 단위 가설 검정

매칭 품질을 통과한 캠페인의 재구매율 효과를 캠페인 단위로 부트스트랩하고, 효과가 0보다 큰지 단측 부호순열 검정을 수행합니다.

In [ ]:
def one_sample_test(values,n_boot=10000,n_perm=20000):
    values=np.asarray(values,float); rng=np.random.default_rng(RANDOM_SEED)
    if len(values)<2: return {"campaigns":len(values),"mean_effect":np.nan,"ci_low":np.nan,"ci_high":np.nan,"p_value_one_sided":np.nan}
    boot=rng.choice(values,size=(n_boot,len(values)),replace=True).mean(axis=1)
    signs=rng.choice([-1,1],size=(n_perm,len(values))); null=(signs*values).mean(axis=1); obs=values.mean()
    return {"campaigns":len(values),"mean_effect":obs,"ci_low":np.quantile(boot,.025),"ci_high":np.quantile(boot,.975),
            "p_value_one_sided":(1+(null>=obs).sum())/(n_perm+1)}

qualified=personalized_coupon_campaign_effects.loc[personalized_coupon_campaign_effects["quality_pass"]].copy()
repurchase_test=one_sample_test(qualified["repurchase_rate_effect"])
revenue_test=one_sample_test(qualified["revenue_28d_effect"])
personalized_coupon_hypothesis_tests=pd.DataFrame([
    {"outcome":"28일 재구매율",**repurchase_test},{"outcome":"28일 매출",**revenue_test}
])
display(personalized_coupon_hypothesis_tests)
supported=(repurchase_test["mean_effect"]>0 and repurchase_test["p_value_one_sided"]<.05)
print("판정: 가설을 지지합니다." if supported else "판정: 현재 데이터로 가설을 지지할 충분한 근거가 없습니다.")

## 6. 해석 주의사항

- 캠페인 배정과 쿠폰 카테고리는 무작위가 아니므로 매칭 후에도 관측되지 않은 차이가 남을 수 있습니다.
- campaign_table은 캠페인 배정을 제공하지만 개별 가구가 캠페인 내 모든 쿠폰을 실제로 받았는지는 직접 확인되지 않습니다. 따라서 ‘캠페인 쿠폰에 선호 카테고리가 포함됨’으로 운영화했습니다.
- 유의한 결과가 나오더라도 무작위 A/B 테스트 없이 완전한 인과효과로 단정하지 않습니다.
- 가설이 지지되지 않으면 카테고리 일치만으로는 충분하지 않고 할인율, 전달 채널, 상품 수준 일치가 중요할 수 있습니다.

In [ ]:
personalized_coupon_campaign_effects.to_csv(OUTPUT_DIR/"personalized_coupon_campaign_effects.csv",index=False,encoding="utf-8-sig")
personalized_coupon_hypothesis_tests.to_csv(OUTPUT_DIR/"personalized_coupon_hypothesis_tests.csv",index=False,encoding="utf-8-sig")
personalized_coupon_pairs.to_csv(OUTPUT_DIR/"personalized_coupon_customer_pairs.csv",index=False,encoding="utf-8-sig")
print("가설 검증 결과 3개 저장 완료")